In [ ]:
from agent_framework import Agent, SkillsProvider, ContextProvider
from agent_framework.openai import OpenAIChatCompletionClient  # for older models (e.g., oss-120b)
import os
from typing import TypedDict
from pathlib import Path

## Base Agent
- No tools
- No skills
- No context providers

In [3]:
chat_client = OpenAIChatCompletionClient(
    base_url=os.getenv("HF_API_BASE_URL"),
    api_key=os.getenv("HF_API_KEY"),
    model="google/gemma-4-31B-it:together",
)

In [5]:
SKILLS_PATH = Path(os.getcwd()).resolve().parent / "skills"
skills_provider = SkillsProvider.from_paths(
    skill_paths=SKILLS_PATH
)

In [68]:
class SkillPathInfo(TypedDict):
    type: str
    skill_path: str

def get_skill_paths(sp: SkillsProvider) -> list[SkillPathInfo]:
    source = sp._source
    found: list[SkillPathInfo] = []
    while source is not None:
        type_name = type(source).__name__
        skill_paths = getattr(source, "_skill_paths", None)
        if skill_paths is not None:
            for skill_path in skill_paths:
                found.append(SkillPathInfo(type=type_name, skill_path=skill_path))

        # Unwrap decorator-style sources
        source = getattr(source, "inner_source", None) or getattr(source, "_inner_source", None) # type: ignore

    return found

In [45]:
agent = Agent(
    client=chat_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[],
    context_providers=[skills_provider],
)

In [70]:
for ctx in agent.context_providers:
    match ctx:
        case SkillsProvider():
            print(get_skill_paths(ctx))


[{'type': 'FileSkillsSource', 'skill_path': '/home/jweston/dev/microsoft-agent-framework-samples/skills'}]


In [ ]:
class AgentInfo(TypedDict):
    name: str
    description: str | None
    instructions: str | None
    model: str | None
    client: str | None

In [19]:
def get_agent_client_type(agent: Agent) -> str:
    return type(agent.client).__name__

In [21]:
def get_agent_info(agent: Agent) -> AgentInfo:
    return AgentInfo(
        name=agent.name,
        description=agent.description,
        instructions=agent.default_options['instructions'],
        model=agent.default_options['model'],
        client=get_agent_client_type(agent)
    )

In [22]:
get_agent_info(agent)

{'name': 'WeatherAgent',
 'description': 'A helpful weather assistant that can answer questions about the weather.',
 'instructions': 'You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. ',
 'model': 'google/gemma-4-31B-it:together',
 'client': 'OpenAIChatCompletionClient'}

In [ ]:
# LEFT-OFF: need to decide if agents should come "prepackaged" with their skills and tools, or if they are added through Noodlr?